# CNN baseline

In [ ]:
# === Paths ===
from pathlib import Path
_ROOT = next((p for p in [Path("/workspace"), Path.cwd(), Path.cwd().parent]
              if (p / "output" / "00_manifest" / "manifest.csv").exists()), Path("/workspace"))
MANIFEST_PATH = str(_ROOT / "output" / "00_manifest" / "manifest.csv")
AUDIO_ROOTS   = [str(_ROOT / "data"), str(_ROOT / "temp_input_files")]
OUTPUT_DIR    = str(_ROOT / "output" / "cnn_baseline")

# === Manifest column mapping ===
MANIFEST_COLS = {
    "clip_id":       "clip_id",
    "audio_path":    "audio_path",
    "filename":      "audio_filename",
    "farm":          "farm",
    "main_category": "main_category_final",
}

# === Audio / spectrogram parameters ===
SAMPLE_RATE   = 16000
N_MELS        = 64
N_FFT         = 1024
HOP_LENGTH    = 512
WIN_LENGTH    = 1024
F_MIN         = 50          # lower bound of the bovine vocal band
F_MAX         = 8000        # full upper half of the 16 kHz Nyquist band
CROP_SECONDS  = 5.0         # fixed crop length used for training and inference windowing
INFERENCE_HOP = 2.5         # stride for sliding-window inference on long clips

# === Training parameters ===
EPOCHS          = 30
BATCH_SIZE      = 16
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
PATIENCE        = 6
RANDOM_SEED     = 42

# === Evaluation tracks ===
RUN_KFOLD = True
RUN_LOFO  = True
K_FOLDS   = 5

# === Targets ===
# These four targets exactly match Layer 2 of the main framework.
TARGETS = ["is_nonvocal", "is_distress_related", "is_resource_related", "proxy_functional_v1"]

# === Framework reference results for the comparison table ===
# Reference macro-F1 and ECE for the framework, per target.
FRAMEWORK_REFERENCE = {
    "is_nonvocal":          {"audio_only_macroF1": 0.780, "lofo_macroF1": 0.649, "raw_ECE": 0.082, "calibrated_ECE": 0.035},
    "is_distress_related":  {"audio_only_macroF1": 0.698, "lofo_macroF1": 0.631, "raw_ECE": 0.227, "calibrated_ECE": 0.121},
    "is_resource_related":  {"audio_only_macroF1": 0.500, "lofo_macroF1": 0.442, "raw_ECE": 0.203, "calibrated_ECE": 0.133},
    "proxy_functional_v1":  {"audio_only_macroF1": 0.457, "lofo_macroF1": 0.409, "raw_ECE": 0.174, "calibrated_ECE": 0.128},
}


In [ ]:
import os, sys, math, time, random, warnings, json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchaudio
import torchaudio.transforms as T

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "layer3_reliability.py").exists():
        sys.path.insert(0, str(_cand))
        break
import layer3_reliability as rel

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, brier_score_loss, log_loss

warnings.filterwarnings("ignore")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest rows: {len(manifest_df)}")
print(f"Manifest columns: {list(manifest_df.columns)}")

# Resolve column names
for k, v in MANIFEST_COLS.items():
    if v not in manifest_df.columns and k in ("clip_id", "farm", "main_category"):
        raise KeyError(f"Required column '{v}' not found in manifest. Update MANIFEST_COLS.")

df = manifest_df.copy()
df["_clip_id"]       = df[MANIFEST_COLS["clip_id"]].astype(str)
df["_farm"]          = df[MANIFEST_COLS["farm"]].astype(str).str.strip()
df["_main_category"] = df[MANIFEST_COLS["main_category"]].astype(str).str.strip()

# Normalize farm labels to {Farm 1, Farm 2, Farm 3}
def _norm_farm(x):
    s = str(x).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits in {"1","2","3"}:
        return f"Farm {digits}"
    return s
df["_farm"] = df["_farm"].map(_norm_farm)

# === Proxy target construction ===
# Proxy-state target definitions used across the project.
PROXY_FUNCTIONAL_MAP = {
    "Non-Vocal Sounds":                    "physiological_nonvocal",
    "Feeding & Hunger Related":            "ingestion_resource",
    "Water & Thirst Related":              "ingestion_resource",
    "Estrus & Mating Behavior":            "reproductive_estrus",
    "Social Recognition & Interaction":    "social_contact",
    "Maternal & Calf Communication":       "social_contact",
    "Distress & Pain":                     "stress_discomfort",
    "Milking & Handling":                  "stress_discomfort",
    "Environmental & Situational":         "stress_discomfort",
}

df["proxy_functional_v1"] = df["_main_category"].map(PROXY_FUNCTIONAL_MAP)
missing_recoding = df[df["proxy_functional_v1"].isna()]
if len(missing_recoding) > 0:
    print(f"WARNING: {len(missing_recoding)} clips have main_category values not in the recoding map.")
    print("Unmapped values:", missing_recoding["_main_category"].unique())
    df = df.dropna(subset=["proxy_functional_v1"]).reset_index(drop=True)

# Binary targets
df["is_nonvocal"]          = (df["proxy_functional_v1"] == "physiological_nonvocal").astype(int)
df["is_distress_related"]  = (df["proxy_functional_v1"] == "stress_discomfort").astype(int)
df["is_resource_related"]  = (df["proxy_functional_v1"] == "ingestion_resource").astype(int)

# Integer encoding for the multiclass target
_classes = sorted(df["proxy_functional_v1"].unique().tolist())
PROXY_FUNCTIONAL_CLASSES = _classes
_cls_to_idx = {c: i for i, c in enumerate(_classes)}
df["proxy_functional_v1_int"] = df["proxy_functional_v1"].map(_cls_to_idx).astype(int)

print("Total clips:", len(df))
print("Per-farm clip counts:")
print(df["_farm"].value_counts())
print("\nProxy-target distributions:")
for t in TARGETS:
    if t == "proxy_functional_v1":
        print(f"  {t}:\n{df['proxy_functional_v1'].value_counts().to_string()}")
    else:
        print(f"  {t}: positives={df[t].sum()}, negatives={len(df)-df[t].sum()}")


In [ ]:
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aif", ".aiff"}

def build_audio_index(roots):
    index = {}
    for root in roots:
        rp = Path(root)
        if not rp.exists():
            continue
        for p in rp.rglob("*"):
            if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
                index.setdefault(p.name.lower(), []).append(str(p))
                index.setdefault(p.stem.lower(), []).append(str(p))
    return index

audio_index = build_audio_index(AUDIO_ROOTS)
print(f"Indexed audio files: {sum(len(v) for v in audio_index.values())}")

def resolve_audio(row):
    # 1) explicit path column
    if MANIFEST_COLS.get("audio_path") in row.index:
        cand = str(row.get(MANIFEST_COLS["audio_path"], "")).strip()
        if cand and Path(cand).exists():
            return cand
    # 2) filename lookup
    fname_col = MANIFEST_COLS.get("filename")
    fname = str(row.get(fname_col, "")).strip() if fname_col in row.index else ""
    if not fname:
        fname = row["_clip_id"]
    matches = audio_index.get(fname.lower(), []) or audio_index.get(Path(fname).stem.lower(), [])
    return matches[0] if matches else ""

df["_audio_path"] = df.apply(resolve_audio, axis=1)
n_resolved = (df["_audio_path"] != "").sum()
print(f"Resolved audio for {n_resolved}/{len(df)} clips.")
if n_resolved < len(df):
    missing = df.loc[df["_audio_path"] == "", "_clip_id"].head(10).tolist()
    print("First missing clip_ids:", missing)
    print("Adjust AUDIO_ROOTS or the manifest filename column and re-run before continuing.")

df = df[df["_audio_path"] != ""].reset_index(drop=True)
print(f"Working set after audio resolution: {len(df)} clips.")


In [ ]:
mel_extractor = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0,
).to(DEVICE)

amp_to_db = T.AmplitudeToDB(top_db=80.0).to(DEVICE)

FRAMES_PER_SEC  = SAMPLE_RATE / HOP_LENGTH                # ~31.25 frames/s at 16 kHz, hop 512
CROP_FRAMES     = int(round(CROP_SECONDS * FRAMES_PER_SEC))
INFERENCE_FRAMES = int(round(INFERENCE_HOP * FRAMES_PER_SEC))

def load_waveform(path):
    """Load mono waveform at SAMPLE_RATE. Returns 1-D float32 tensor."""
    wav, sr = torchaudio.load(path)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = T.Resample(sr, SAMPLE_RATE)(wav)
    wav = wav.squeeze(0)
    # peak normalize to reduce gain mismatch across recorders
    peak = wav.abs().max()
    if peak > 0:
        wav = wav / peak
    return wav.float()

def waveform_to_logmel(wav_tensor):
    """Convert a 1-D waveform tensor to a 2-D log-mel-spectrogram (n_mels x T)."""
    with torch.no_grad():
        mel = mel_extractor(wav_tensor.to(DEVICE).unsqueeze(0))   # 1 x n_mels x T
        logmel = amp_to_db(mel).squeeze(0).cpu()                  # n_mels x T
    return logmel

# Pre-extract log-mel-spectrograms for every clip once (avoids re-decoding audio every epoch).
print("Pre-extracting log-mel spectrograms ...")
logmel_cache = {}
t0 = time.time()
for i, row in df.iterrows():
    cid = row["_clip_id"]
    try:
        wav = load_waveform(row["_audio_path"])
        logmel = waveform_to_logmel(wav)
    except Exception as e:
        print(f"  failed {cid}: {e}")
        continue
    logmel_cache[cid] = logmel
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(df)} clips ({time.time()-t0:.1f}s)")
print(f"Cached {len(logmel_cache)} spectrograms in {time.time()-t0:.1f}s.")

df = df[df["_clip_id"].isin(logmel_cache.keys())].reset_index(drop=True)
print(f"Final working set: {len(df)} clips.")


In [ ]:
def crop_or_pad(logmel, target_frames):
    n_frames = logmel.size(-1)
    if n_frames >= target_frames:
        start = random.randint(0, n_frames - target_frames)
        return logmel[:, start:start+target_frames]
    pad_total = target_frames - n_frames
    pad_left  = pad_total // 2
    pad_right = pad_total - pad_left
    return F.pad(logmel.unsqueeze(0), (pad_left, pad_right), mode="reflect").squeeze(0)

def crop_windows_for_inference(logmel, win_frames, hop_frames):
    n_frames = logmel.size(-1)
    if n_frames <= win_frames:
        return [crop_or_pad(logmel, win_frames)]
    out = []
    start = 0
    while start + win_frames <= n_frames:
        out.append(logmel[:, start:start+win_frames])
        start += hop_frames
    if start < n_frames and (n_frames - start) > win_frames // 2:
        out.append(logmel[:, n_frames-win_frames:n_frames])
    return out if out else [crop_or_pad(logmel, win_frames)]

class ClipDataset(Dataset):
    def __init__(self, df_subset, target_col, multiclass=False, mode="train"):
        self.df = df_subset.reset_index(drop=True)
        self.target_col = target_col
        self.multiclass = multiclass
        self.mode = mode
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cid = row["_clip_id"]
        logmel = logmel_cache[cid]
        x = crop_or_pad(logmel, CROP_FRAMES)
        # per-clip standardization for stability
        x = (x - x.mean()) / (x.std() + 1e-6)
        x = x.unsqueeze(0)  # 1 x n_mels x CROP_FRAMES
        if self.multiclass:
            y = int(row["proxy_functional_v1_int"])
            return x, y
        else:
            y = float(row[self.target_col])
            return x, y


In [ ]:
class MelCNN(nn.Module):
    def __init__(self, n_classes=2, dropout=0.3):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.feat = nn.Sequential(
            block(1, 32),
            block(32, 64),
            block(64, 128),
            block(128, 128),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.head(self.feat(x))


In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=15):
    """Top-label ECE for binary or multiclass."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if y_prob.ndim == 1:
        confidences = np.where(y_prob >= 0.5, y_prob, 1 - y_prob)
        predictions = (y_prob >= 0.5).astype(int)
    else:
        confidences = y_prob.max(axis=1)
        predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() > 0:
            ece += (mask.sum() / n) * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)

def multiclass_brier(y_true, y_prob, n_classes):
    onehot = np.zeros((len(y_true), n_classes))
    onehot[np.arange(len(y_true)), np.asarray(y_true).astype(int)] = 1
    return float(np.mean(np.sum((y_prob - onehot) ** 2, axis=1)))

def predict_clip_logmel(model, logmel, multiclass):
    """Sliding-window inference over the full clip; average softmax/sigmoid."""
    model.eval()
    windows = crop_windows_for_inference(logmel, CROP_FRAMES, INFERENCE_FRAMES)
    probs = []
    with torch.no_grad():
        for w in windows:
            w = (w - w.mean()) / (w.std() + 1e-6)
            x = w.unsqueeze(0).unsqueeze(0).to(DEVICE)
            logits = model(x)
            if multiclass:
                p = torch.softmax(logits, dim=1).cpu().numpy()[0]
            else:
                p = torch.softmax(logits, dim=1).cpu().numpy()[0, 1]  # P(positive class)
            probs.append(p)
    return np.mean(probs, axis=0)

def train_one_run(train_df, val_df, target, multiclass, epochs=EPOCHS, patience=PATIENCE):
    n_classes = len(PROXY_FUNCTIONAL_CLASSES) if multiclass else 2
    model = MelCNN(n_classes=n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    if multiclass:
        cls_counts = train_df["proxy_functional_v1_int"].value_counts().reindex(range(n_classes), fill_value=1)
        weights = torch.tensor((1.0 / cls_counts.values).astype(np.float32) * n_classes / (1.0 / cls_counts.values).sum())
        weights = weights.to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weights)
    else:
        n_pos = train_df[target].sum(); n_neg = len(train_df) - n_pos
        weights = torch.tensor([1.0 / max(n_neg,1), 1.0 / max(n_pos,1)], dtype=torch.float32, device=DEVICE)
        weights = weights * 2 / weights.sum()
        criterion = nn.CrossEntropyLoss(weight=weights)

    train_loader = DataLoader(
        ClipDataset(train_df, target, multiclass=multiclass, mode="train"),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0,
    )

    best_f1 = -1.0
    best_state = None
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(DEVICE)
            if multiclass:
                y = y.long().to(DEVICE)
            else:
                y = y.long().to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        # validate using clip-level sliding-window prediction
        y_true, y_prob = [], []
        for _, row in val_df.iterrows():
            p = predict_clip_logmel(model, logmel_cache[row["_clip_id"]], multiclass)
            if multiclass:
                y_true.append(int(row["proxy_functional_v1_int"]))
                y_prob.append(p)
            else:
                y_true.append(int(row[target]))
                y_prob.append(p)
        y_true = np.asarray(y_true)
        y_prob = np.asarray(y_prob)
        y_pred = y_prob.argmax(axis=1) if multiclass else (y_prob >= 0.5).astype(int)
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # final validation predictions from best model
    y_true, y_prob = [], []
    for _, row in val_df.iterrows():
        p = predict_clip_logmel(model, logmel_cache[row["_clip_id"]], multiclass)
        if multiclass:
            y_true.append(int(row["proxy_functional_v1_int"]))
            y_prob.append(p)
        else:
            y_true.append(int(row[target]))
            y_prob.append(p)
    return np.asarray(y_true), np.asarray(y_prob)

def fold_metrics(y_true, y_prob, multiclass, n_classes=None):
    if multiclass:
        y_pred = y_prob.argmax(axis=1)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        bal_acc  = balanced_accuracy_score(y_true, y_pred)
        ece      = expected_calibration_error(y_true, y_prob)
        brier    = multiclass_brier(y_true, y_prob, n_classes)
    else:
        y_pred = (y_prob >= 0.5).astype(int)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        bal_acc  = balanced_accuracy_score(y_true, y_pred)
        ece      = expected_calibration_error(y_true, y_prob)
        brier    = brier_score_loss(y_true, y_prob)
    return {"macro_f1": macro_f1, "balanced_accuracy": bal_acc, "ECE": ece, "Brier": brier}


In [ ]:
all_results = []

if RUN_KFOLD:
    for target in TARGETS:
        multiclass = (target == "proxy_functional_v1")
        strat_col = "proxy_functional_v1_int" if multiclass else target
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        print(f"\n=== 5-fold CV on target: {target} ===")
        for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(df, df[strat_col])):
            train_df = df.iloc[tr_idx].copy()
            val_df   = df.iloc[va_idx].copy()
            t0 = time.time()
            y_true, y_prob = train_one_run(train_df, val_df, target, multiclass)
            m = fold_metrics(y_true, y_prob, multiclass, n_classes=len(PROXY_FUNCTIONAL_CLASSES))
            row = {"track": "5fold_audio_only", "target": target, "fold": fold_idx, **m,
                   "n_train": len(train_df), "n_val": len(val_df), "sec": round(time.time()-t0,1)}
            all_results.append(row)
            print(f"  fold {fold_idx}: macroF1={m['macro_f1']:.3f} balAcc={m['balanced_accuracy']:.3f} "
                  f"ECE={m['ECE']:.3f} Brier={m['Brier']:.3f} ({row['sec']}s)")


In [ ]:
if RUN_LOFO:
    cnn_lofo_preds = []
    farm_df = df[df["_farm"].isin(["Farm 1", "Farm 2", "Farm 3"])].reset_index(drop=True)
    print(f"LOFO uses {len(farm_df)} farm-identified clips.")
    farms = sorted(farm_df["_farm"].unique())

    for target in TARGETS:
        multiclass = (target == "proxy_functional_v1")
        print(f"\n=== LOFO on target: {target} ===")
        for held_farm in farms:
            train_df = farm_df[farm_df["_farm"] != held_farm].copy()
            val_df   = farm_df[farm_df["_farm"] == held_farm].copy()
            # ensure at least one positive in val for binary targets
            if not multiclass and val_df[target].sum() == 0:
                print(f"  skip held farm {held_farm}: no positives")
                continue
            t0 = time.time()
            y_true, y_prob = train_one_run(train_df, val_df, target, multiclass)
            m = fold_metrics(y_true, y_prob, multiclass, n_classes=len(PROXY_FUNCTIONAL_CLASSES))
            for _j, _cid in enumerate(val_df["_clip_id"].tolist()):
                _rec = {"target": target, "held_farm": held_farm, "_record_key": _cid,
                        "task_type": "multiclass" if multiclass else "binary", "y_true": int(y_true[_j])}
                if multiclass:
                    for _k, _cls in enumerate(PROXY_FUNCTIONAL_CLASSES):
                        _rec[f"proba__{_cls}"] = float(y_prob[_j, _k])
                else:
                    _rec["proba__1"] = float(y_prob[_j])
                cnn_lofo_preds.append(_rec)
            row = {"track": "LOFO", "target": target, "held_farm": held_farm, **m,
                   "n_train": len(train_df), "n_val": len(val_df), "sec": round(time.time()-t0,1)}
            all_results.append(row)
            print(f"  held {held_farm}: macroF1={m['macro_f1']:.3f} balAcc={m['balanced_accuracy']:.3f} "
                  f"ECE={m['ECE']:.3f} Brier={m['Brier']:.3f} ({row['sec']}s)")


In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(Path(OUTPUT_DIR) / "cnn_baseline_per_fold.csv", index=False)
print("Saved per-fold results.")

# Pooled summary: mean ± std across folds for each (track, target)
summary = (
    results_df
    .groupby(["track", "target"])[["macro_f1", "balanced_accuracy", "ECE", "Brier"]]
    .agg(["mean", "std"])
    .round(3)
)
summary.to_csv(Path(OUTPUT_DIR) / "cnn_baseline_summary.csv")
print("\n=== CNN baseline summary (mean ± std across folds) ===")
print(summary)

# Side-by-side comparison against framework numbers
comp_rows = []
for target in TARGETS:
    fw = FRAMEWORK_REFERENCE[target]
    cnn_kfold = results_df.query("track == '5fold_audio_only' and target == @target")
    cnn_lofo  = results_df.query("track == 'LOFO' and target == @target")
    comp_rows.append({
        "target": target,
        "framework_audio_only_macroF1": fw["audio_only_macroF1"],
        "cnn_5fold_macroF1_mean":       round(cnn_kfold["macro_f1"].mean(), 3) if len(cnn_kfold) else np.nan,
        "cnn_5fold_macroF1_std":        round(cnn_kfold["macro_f1"].std(),  3) if len(cnn_kfold) else np.nan,
        "framework_LOFO_macroF1":       fw["lofo_macroF1"],
        "cnn_LOFO_macroF1_mean":        round(cnn_lofo["macro_f1"].mean(),  3) if len(cnn_lofo) else np.nan,
        "cnn_LOFO_macroF1_std":         round(cnn_lofo["macro_f1"].std(),   3) if len(cnn_lofo) else np.nan,
        "framework_raw_ECE":            fw["raw_ECE"],
        "framework_calibrated_ECE":     fw["calibrated_ECE"],
        "cnn_LOFO_raw_ECE_mean":        round(cnn_lofo["ECE"].mean(), 3) if len(cnn_lofo) else np.nan,
    })
comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(Path(OUTPUT_DIR) / "framework_vs_cnn_comparison.csv", index=False)
print("\n=== Framework vs CNN baseline comparison ===")
with pd.option_context('display.max_columns', None, 'display.width', 220):
    print(comp_df.to_string(index=False))

print(f"\nAll outputs written to: {OUTPUT_DIR}")
print("  - cnn_baseline_per_fold.csv")
print("  - cnn_baseline_summary.csv")
print("  - framework_vs_cnn_comparison.csv")


In [ ]:
COVERAGES = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
cnn_pred = pd.DataFrame(cnn_lofo_preds)
cnn_pred.to_csv(Path(OUTPUT_DIR) / "cnn_lofo_predictions.csv.gz", index=False, compression="gzip")


def _composite(rows, keys=("ece", "brier", "logloss")):
    ranked = {k: sorted(range(len(rows)), key=lambda i: rows[i][k]) for k in keys}
    score = [0] * len(rows)
    for k in keys:
        for r, i in enumerate(ranked[k]):
            score[i] += r
    return int(np.argmin(score))


rel_rows = []
for target in TARGETS:
    sub = cnn_pred[cnn_pred.target == target].reset_index(drop=True)
    task = sub.task_type.iloc[0]
    farms = sub.held_farm.to_numpy()
    y = sub.y_true.to_numpy().astype(int)
    if task == "binary":
        P = np.c_[1 - sub["proba__1"].to_numpy(), sub["proba__1"].to_numpy()]
        n_classes = 2
        methods = rel.BINARY_METHODS
    else:
        cols = [f"proba__{c}" for c in PROXY_FUNCTIONAL_CLASSES]
        P = sub[cols].to_numpy()
        n_classes = len(cols)
        methods = rel.MULTICLASS_METHODS

    candidates = []
    for method in methods:
        if task == "binary":
            cp1 = rel.group_aware_binary(farms, P[:, 1], y, method)
            CP = np.c_[1 - cp1, cp1]
        else:
            CP = rel.group_aware_multiclass(farms, P, y, method, n_classes)
        candidates.append({
            "method": method,
            "ece": rel.expected_calibration_error(y, CP.max(1), CP.argmax(1)),
            "brier": rel.brier_binary(y, CP[:, 1]) if task == "binary" else rel.brier_multiclass(y, CP, n_classes),
            "logloss": rel.log_loss_safe(y, CP, n_classes),
            "CP": CP,
        })
    raw = candidates[0]
    best = candidates[_composite(candidates)]

    best80, best_tag, best_base = None, None, None
    for tag, CP in {"raw": raw["CP"], "calibrated": best["CP"]}.items():
        r80 = next(r for r in rel.selective_prediction(y, CP.argmax(1), CP.max(1), COVERAGES)
                   if r["coverage_target"] == 0.80)
        base = f1_score(y, CP.argmax(1), average="macro", zero_division=0)
        if best80 is None or r80["retained_macro_f1"] > best80["retained_macro_f1"]:
            best80, best_tag, best_base = r80, tag, base

    rel_rows.append({
        "target": target,
        "cnn_lofo_macro_f1": round(f1_score(y, P.argmax(1), average="macro", zero_division=0), 3),
        "cnn_ece_raw": round(raw["ece"], 3),
        "cnn_ece_calibrated": round(best["ece"], 3),
        "cnn_calibration_method": best["method"],
        "cnn_retained_macro_f1_80cov": round(best80["retained_macro_f1"], 3),
        "cnn_abstention_rate": round(best80["abstention_rate"], 3),
        "cnn_selective_score": best_tag,
    })

cnn_reliability = pd.DataFrame(rel_rows)
cnn_reliability.to_csv(Path(OUTPUT_DIR) / "cnn_reliability_summary.csv", index=False)
print(cnn_reliability.to_string(index=False))